In [0]:
-- Ejercicio 0.1: CTE básica - Estadísticas por zona

WITH estadisticas_por_zona AS (
    SELECT
        pickup_zip AS trip_zone,
        ROUND(AVG(trip_distance), 2) AS avg_trip_distance,
        ROUND(AVG(fare_amount), 2) AS avg_fare_amount
    FROM samples.nyctaxi.trips
    GROUP BY pickup_zip
    HAVING COUNT(*) > 100
)
SELECT
    trip_zone,
    avg_trip_distance,
    avg_fare_amount
FROM estadisticas_por_zona
ORDER BY trip_zone ASC;

In [0]:
-- Ejercicio 0.2: CTE múltiple - Análisis comparativo
WITH promedio_por_hora AS (
    SELECT
        HOUR(tpep_pickup_datetime) AS hora_dia,
        AVG(fare_amount) AS promedio_tarifa
    FROM samples.nyctaxi.trips
    GROUP BY HOUR(tpep_pickup_datetime)
),
promedio_general AS (
    SELECT
        AVG(fare_amount) AS promedio_tarifa
    FROM samples.nyctaxi.trips
)
SELECT 
    ph.hora_dia,
    ph.promedio_tarifa,
    pg.promedio_tarifa AS promedio_general_tarifa,
    (ph.promedio_tarifa - pg.promedio_tarifa) AS diferencia
FROM promedio_por_hora AS ph
JOIN promedio_general AS pg
ORDER BY hora_dia;

In [0]:
-- Ejercicio 0.3: CTE para limpieza de datos
WITH viajes_validos AS (
    SELECT *
    FROM samples.nyctaxi.trips
    WHERE trip_distance > 0
        AND fare_amount > 0
)
SELECT
    MIN(trip_distance) AS distancia_minima,
    MAX(trip_distance) AS distancia_maxima,
    AVG(trip_distance) AS distancia_promedio,
    MEDIAN(trip_distance) AS distancia_mediana
FROM viajes_validos;

In [0]:
-- Ejercicio 0.4: ROW_NUMBER() para ranking
SELECT 
    tpep_pickup_datetime,
    trip_distance,
    fare_amount,
    ROW_NUMBER() OVER(ORDER BY fare_amount DESC) AS ranking
FROM samples.nyctaxi.trips
LIMIT 20;

In [0]:
-- Ejercicio 0.5: Window Function - ROW_NUMBER() vs RANK() vs DENSE_RANK()
SELECT 
    tpep_pickup_datetime,
    trip_distance,
    fare_amount,
    ROW_NUMBER() OVER(ORDER BY fare_amount DESC) AS ranking_row,
    RANK() OVER(ORDER BY fare_amount DESC) AS ranking_rank,
    DENSE_RANK() OVER(ORDER BY fare_amount DESC) AS ranking_dr
FROM samples.nyctaxi.trips
LIMIT 20;

In [0]:
-- Ejercicio 0.6: Comparar con promedio general
SELECT
    fare_amount AS tarifa,
    trip_distance AS distancia,
    AVG(fare_amount) OVER() AS tarifa_promedio_general,
    tarifa - tarifa_promedio_general AS diferencia,
    ROUND(
        (ABS(tarifa - tarifa_promedio_general) / ((tarifa + tarifa_promedio_general) / 2)) * 100, 
        2
    ) AS porcentaje_diferencia
FROM samples.nyctaxi.trips

In [0]:
-- Ejercicio 0.7: Promedio por partición
SELECT 
    tpep_pickup_datetime AS fecha,
    fare_amount AS tarifa,
    pickup_zip AS zona_de_inicio,
    AVG(fare_amount) OVER(PARTITION BY pickup_zip) AS promedio_tarifa_zona
FROM samples.nyctaxi.trips

In [0]:
-- Ejercicio 0.8: LAG() para comparar con anterior
SELECT
    tpep_pickup_datetime AS fecha,
    fare_amount AS tarifa,
    LAG(fare_amount) OVER(ORDER BY tpep_pickup_datetime) AS tarifa_viaje_anterior
FROM samples.nyctaxi.trips

In [0]:
-- Ejercicio 0.9: SUM() acumulado
SELECT
    tpep_pickup_datetime AS fecha,
    fare_amount AS tarifa,
    SUM(fare_amount) 
        OVER(ORDER BY tpep_pickup_datetime ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS total_acumulado 
FROM samples.nyctaxi.trips

In [0]:
-- Ejercicio 0.10: Combinando CTEs y Window Functions
WITH viajes_validos AS (
    SELECT *
    FROM samples.nyctaxi.trips
    WHERE trip_distance > 0
        AND fare_amount > 0
),
estadisticas_por_zona AS (
    SELECT 
        pickup_zip,
        ROUND(AVG(fare_amount), 2) AS promedio_tarifa,
        MAX(fare_amount) AS tarifa_maxima,
        MIN(fare_amount) AS tarifa_minima
    FROM viajes_validos
    GROUP BY pickup_zip
)
SELECT
    pickup_zip AS zona,
    promedio_tarifa,
    tarifa_maxima,
    tarifa_minima,
    ROW_NUMBER() OVER(ORDER BY promedio_tarifa DESC) AS ranking_tarifa 
FROM estadisticas_por_zona
LIMIT 10;